### Градиентные методы многомерной минимизации
К основным градиентным методам многомерной минимизации относят:
1. метод градиентного спуска с постоянным шагом;
2. метод наискорейшего градиентного спуска;
3. овражный метод.

Основными численными методами многомерной минимизации второго порядка
являются:
1. метод Ньютона;
2. метод Метод Ньютона с регулировкой шага;
3. метод Левенберга-Марквардта.

#### Вариант заданий для поиска минимума градиентным методом
|Номер|градиент метод|функция|
| --- | --- | --- |
|13|1,2|f(x1, x2) = x1^2-x1x2+x2^2+3x1-2x2+1|

#### Вариант заданий для поиска минимума методами второго порядка
|Номер|Метод 2-го порядка|функция|
| --- | --- | --- |
|13|1|f(x1,x2)=100(1-x1^2-x2^2)^2+(x1-1)^2+x2^2|

Для градиентных методов исследовать сходимость алгоритма, фиксируя найденное решение, значение функции, количество итераций метода и количество вычислений функции в зависимости от задаваемой точности поиска (ε = 10−2, 10−4, 10−6). Результаты оформить в виде таблицы:

Для метода второго порядка исследовать сходимость алгоритма по числу итераций при выборе различных начальных точек (не менее трёх). Результаты оформить в виде таблицы:

In [50]:
import numpy as np
import sympy as sp
from scipy.optimize import minimize_scalar
import plotly.graph_objects as go

def csgradM(f, x0, e1, e2, gamma, M):
    x = x0
    x1, x2 = sp.symbols('x1, x2')
    fun = sp.lambdify([x1, x2], f, "math")

    trajectory = [x0]
    values = [fun(x[0], x[1])]

    k = 0
    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")

    c = 0

    while k < M:
        c_grad = np.array([x1d(x[0], x[1]), x2d(x[0], x[1])])
        if np.linalg.norm(c_grad) < e1:
            break
        x_next = x - (gamma * c_grad)
        while fun(x_next[0], x_next[1]) - fun(x[0], x[1]) >= 0 and gamma > 1e-10:
            gamma = gamma / 2
            x_next = x - (gamma * c_grad)
        if np.linalg.norm(x_next - x) <= e2 and abs(fun(x_next[0], x_next[1]) - fun(x[0], x[1])) <= e2:
            c += 1
            x = x_next
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
            if c == 2:
                break
        else:
            c = 0
            x = x_next
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
        k += 1
    return np.array(trajectory), np.array(values), k

def qgradM(f, x0, e1, e2, gamma, M):
    def g_calc(gamma):
            x_new = x - gamma * c_grad
            return fun(x_new[0], x_new[1])
    x = x0
    x1, x2 = sp.symbols('x1, x2')
    fun = sp.lambdify([x1, x2], f, "math")

    trajectory = [x0]
    values = [fun(x[0], x[1])]

    

    k = 0
    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")

    c = 0

    while k < M:
        c_grad = np.array([x1d(x[0], x[1]), x2d(x[0], x[1])])
        if np.linalg.norm(c_grad) < e1:
            break
        
        gamma = minimize_scalar(g_calc, bounds=(0, 1e3), method='bounded')
        gamma = gamma.x
        x_next = x - (gamma * c_grad)
        if np.linalg.norm(x_next - x) <= e2 and abs(fun(x_next[0], x_next[1]) - fun(x[0], x[1])) <= e2:
            c += 1
            c += 1
            x = x_next
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
            if c == 2:
                break
        else:
            c = 0
            x = x_next
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
        k += 1
    return np.array(trajectory), np.array(values), k

e_list = [10**(-2), 10**(-4), 10**(-6)]

x1, x2 = sp.symbols('x1, x2')
f = x1**2-x1*x2+x2**2+3*x1-2*x2+1
np.random.seed(42)
x0 = np.random.uniform(-10, 10, size=2)

x1_range = np.linspace(-10, 10, 100)
x2_range = np.linspace(-10, 10, 100)
x, y = np.meshgrid(x1_range, x2_range)
fu = x**2-x*y+y**2+3*x-2*y+1
fig = go.Figure(data=[go.Contour(
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
        coloring = 'heatmap',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'x1**2-x1*x2+x2**2+3*x1-2*x2+1') 

for e in e_list:
    print(e)
    ans = csgradM(f, x0=x0, e1 = e, e2 = e, gamma = 1, M = 50)
    print(ans[0][-1], ans[1][-1], ans[2])
    
    fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'для ε = {e}'))
    fig.update_layout(
    legend=dict(
        orientation="h",   
        yanchor="bottom",
        y=1.02,            
        xanchor="right",
        x=1               
    )
)
fig.show()

fig1 = go.Figure(data=[go.Contour(
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
        coloring = 'heatmap',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig1.update_layout(title=f'x1**2-x1*x2+x2**2+3*x1-2*x2+1') 
for e in e_list:
    print(e)
    ans = qgradM(f, x0=x0, e1 = e, e2 = e, gamma = 1, M = 50)
    print(ans[0][-1], ans[1][-1], ans[2])
    
    fig1.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'для ε = {e}'))
    fig1.update_layout(
    legend=dict(
        orientation="h",   
        yanchor="bottom",
        y=1.02,            
        xanchor="right",
        x=1               
    )
)
fig1.show()

0.01
[-1.33362041  0.33545271] -1.3333281507559231 12
0.0001
[-1.33333782  0.33336645] -1.3333333320680563 18
1e-06
[-1.33333307  0.3333333 ] -1.3333333333332562 25


0.01
[-1.33374476  0.33637071] -1.3333226887848642 6
0.0001
[-1.33326784  0.3333886 ] -1.3333333296094052 9
1e-06
[-1.33333301  0.33333361] -1.3333333333332402 12


In [ ]:
import numpy as np
import sympy as sp
import plotly.graph_objects as go

def newton(f, x0, e1, e2, gamma, M):
    x = x0
    x1, x2 = sp.symbols('x1, x2')
    fun = sp.lambdify([x1, x2], f, "math")

    trajectory = [x0]
    values = [fun(x[0], x[1])]

    k = 0
    x1d = sp.diff(f, x1)
    x2d = sp.diff(f, x2)
    hessian = sp.hessian(f, [x1, x2])
    x1d = sp.lambdify([x1, x2], x1d, "math")
    x2d = sp.lambdify([x1, x2], x2d, "math")
    t = 1
    c = 0
    x = x.astype(float)
    while k < M:
        c_grad = np.array([x1d(x[0], x[1]), x2d(x[0], x[1])])
        if np.linalg.norm(c_grad) < e1:
            break
        H = np.array(hessian.subs({x1: x[0], x2: x[1]})).astype(float)
        
        # Check if Hessian is positive definite, if not, use gradient descent
        if np.linalg.det(H) <= 0 or np.any(np.linalg.eigvals(H) <= 0):
            d = -gamma * c_grad  # fallback to gradient descent
        else:
            H_inv = np.linalg.inv(H)
            d = -H_inv * c_grad

        x_next = (x + t*d).astype(float)
        if np.linalg.norm(x_next - x) <= e2 and abs(fun(x_next[0], x_next[1]) - fun(x[0], x[1])) <= e2:
            c += 1
            trajectory.append(x)
            values.append(fun(x[0], x[1]))
            if c == 2:
                break
        else:
            c = 0
            k += 1
            x = x_next
            trajectory.append(x)
            values.append(fun(x[0], x[1]))

    return np.array(trajectory), np.array(values), k


x1, x2 = sp.symbols('x1, x2')
f = 100*(1 - x1**2 - x2**2)**2 + (x1-1)**2 + x2**2
np.random.seed(42)
p_list = (np.random.uniform(-3, 3, size=2), np.random.uniform(-3, 3, size=2), np.random.uniform(-3, 3, size=2))

x1_range = np.linspace(-10, 10, 100)
x2_range = np.linspace(-10, 10, 100)
x, y = np.meshgrid(x1_range, x2_range)
fu = 100*(1 - x**2 - y**2)**2 + (x-1)**2 + y**2
fig = go.Figure(data=[go.Contour(
    zmax = 5000, zmin = 0,
    z=fu, x=x1_range, y=x2_range, colorscale='Jet',
    contours=dict(
         coloring = 'heatmap',
        showlabels = True,
        labelfont = dict(
            size = 12,
            color = 'white',
        )
    ) 
)])
fig.update_layout(title=f'100(1-x1^2-x2^2)^2+(x1-1)^2+x2^2') 

for point in p_list:
    ans = newton(f, x0=point, e1 = 0.000001, e2 = 0.000001, gamma = 1, M = 50)
    print(ans[0][-1], ans[1][-1], ans[2])
    
    fig.add_trace(go.Scatter(x=[i[0] for i in ans[0]], y=[i[1] for i in ans[0]], name=f'точка {point.tolist()}'))
    fig.update_layout(
    legend=dict(
        orientation="h",   
        yanchor="bottom",
        y=1.02,            
        xanchor="right",
        x=1               
    )
)
fig.show()

[1.25031326e-03 2.23561883e+00] 1604.3903771854891 50
[0.00307185 1.62182205] 269.41726223650875 50
[ 5.22499634e-04 -3.24993036e+00] 9154.836510215075 50
